# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [14]:
import os
from glob import glob

# Write your code below.
price_data_path = os.getenv("PRICE_DATA")
file_paths = glob(os.path.join(price_data_path, "**/*.parquet"), recursive=True)
print("Found files:", file_paths)



Found files: ['../../05_src/data/prices/LOGC/LOGC_2018/part.0.parquet', '../../05_src/data/prices/LOGC/LOGC_2018/part.1.parquet', '../../05_src/data/prices/LOGC/LOGC_2020/part.0.parquet', '../../05_src/data/prices/LOGC/LOGC_2020/part.1.parquet', '../../05_src/data/prices/LOGC/LOGC_2019/part.0.parquet', '../../05_src/data/prices/LOGC/LOGC_2019/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2012/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2012/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2015/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2015/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2014/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2014/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2013/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2013/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1980/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_1980/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1987/part.0.parquet', 

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Create lagged values for Close and Adj_Close
df['Close_lag_1'] = df.groupby('ticker')['Close'].shift(1)
df['Adj_Close_lag_1'] = df.groupby('ticker')['Adj_Close'].shift(1)

# Calculate returns
df['returns'] = (df['Close'] / df['Close_lag_1']) - 1

# Calculate high-low range
df['hi_lo_range'] = df['High'] - df['Low']

dd_feat = df

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [92]:
df_pandas = dd_feat.copy()

# Add 10-day rolling average of returns
df_pandas['returns_ma_10'] = (
    df_pandas
    .groupby('ticker')['returns']
    .transform(lambda x: x.rolling(window=10, min_periods=1).mean())
)

df_pandas[['Date', 'ticker', 'returns', 'returns_ma_10']].head(11)


,Date,ticker,returns,returns_ma_10
0,2018-10-19,LOGC,NaN,NaN
1,2018-10-22,LOGC,-0.039130,-0.039130
2,2018-10-23,LOGC,-0.085973,-0.062552
3,2018-10-24,LOGC,-0.219802,-0.114968
4,2018-10-25,LOGC,0.012690,-0.083054
5,2018-10-26,LOGC,0.075188,-0.051405
6,2018-10-29,LOGC,0.250583,-0.001074
7,2018-10-30,LOGC,0.021435,0.002142
8,2018-10-31,LOGC,-0.024635,-0.001206
9,2018-11-01,LOGC,0.038354,0.003190


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It is not necessary but since this is a smaller datasets, converting to panda is easier and more efficient

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.